In [17]:
YEAR_FILTER = '0113-2'  # 修改為你想篩選的年份
data_path = f'all_predictions_{YEAR_FILTER}.csv'  # 替換為你的數據文件路徑
true_path = f'true_value_{YEAR_FILTER}.csv'

In [18]:
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, mean_squared_error, mean_absolute_error

def evaluate_popularity_classification(data_path, true_path, true_label_column, predicted_label_column):
    """
    評估熱門度分類的模型表現。
    :param data_path: 包含真實標籤與預測標籤的數據文件路徑
    :param true_label_column: 真實標籤列名
    :param predicted_label_column: 預測標籤列名
    :return: 評估指標與混淆矩陣
    """
    # 讀取數據
    df = pd.read_csv(data_path)
    tr = pd.read_csv(true_path)

    # 確保所需列存在
    if true_label_column not in tr.columns or predicted_label_column not in df.columns:
        raise ValueError(f"數據中缺少必要列：{true_label_column} 或 {predicted_label_column}")

    # 提取真實標籤與預測標籤
    y_true = tr[true_label_column]
    y_pred = df[predicted_label_column]

    # 計算混淆矩陣
    conf_matrix = confusion_matrix(y_true, y_pred, labels=['高', '中', '低'])

    # 計算評估指標
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    # 輸出結果
    print("熱門度分類評估：")
    print(f"混淆矩陣：\n{conf_matrix}")
    print(f"準確率 (Accuracy): {accuracy:.2f}")
    print(f"召回率 (Recall): {recall:.2f}")
    print(f"F1 分數: {f1:.2f}")

    return conf_matrix, accuracy, recall, f1

# 使用範例

evaluate_popularity_classification(data_path, true_path, true_label_column='最佳標籤', predicted_label_column='最佳標籤')

熱門度分類評估：
混淆矩陣：
[[ 8 16  8]
 [10 11  2]
 [ 3  7  9]]
準確率 (Accuracy): 0.38
召回率 (Recall): 0.38
F1 分數: 0.37


(array([[ 8, 16,  8],
        [10, 11,  2],
        [ 3,  7,  9]], dtype=int64),
 0.3783783783783784,
 0.3783783783783784,
 0.37212932893171224)

In [19]:
def evaluate_regression(predict_path, true_value_path, output_file):
    """根據系所和課程名稱計算預測選課人數的回歸指標"""
    # 讀取資料
    predict_df = pd.read_csv(predict_path)
    true_value_df = pd.read_csv(true_value_path)

    # 確保必須的欄位存在
    required_columns = ['系所', '科目名稱', '預測選課']
    if not all(col in predict_df.columns for col in required_columns):
        raise ValueError(f"預測資料缺少必要列：{required_columns}")
    
    # 修改欄位名稱
    true_value_df.rename(columns={
        '預測選課': '已選人數',
        '預測餘額': '剩餘名額',
        # 根據需要修改更多欄位名稱
    }, inplace=True)
    
    required_columns_true = ['系所', '科目名稱', '已選人數']
    if not all(col in true_value_df.columns for col in required_columns_true):
        raise ValueError(f"真實資料缺少必要列：{required_columns_true}")

    # 合併資料，根據 '系所' 和 '課程名稱' 對應
    combined_df = pd.merge(
        predict_df, 
        true_value_df, 
        on=['系所', '科目名稱'], 
        how='inner'
    )
    
    if combined_df.empty:
        raise ValueError("無法找到匹配的系所和課程名稱，請檢查資料一致性。")

    # 提取真實值和預測值
    y_true = combined_df['已選人數']  # 真實選課人數
    y_pred = combined_df['預測選課']  # 預測選課人數

    # 計算回歸指標
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    print(f"回歸模型評估：")
    print(f"均方誤差 (MSE): {mse:.2f}")
    print(f"平均絕對誤差 (MAE): {mae:.2f}")

    # 儲存合併結果和指標
    combined_df['誤差'] = y_pred - y_true  # 計算誤差
    combined_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"已將合併結果和誤差儲存至 {output_file}")

    return mse, mae

if __name__ == '__main__':
    OUTPUT_FILE = f'evaluation_results_{YEAR_FILTER}.csv'  # 合併後的輸出檔案

    # 執行回歸評估
    evaluate_regression(data_path, true_path, OUTPUT_FILE)

回歸模型評估：
均方誤差 (MSE): 909.80
平均絕對誤差 (MAE): 19.55
已將合併結果和誤差儲存至 evaluation_results_0113-2.csv


In [20]:
def filter_and_combine_high_labels(file1, file2, output_file):
    """讀取兩個 CSV，篩選最佳標籤為 '高' 的資料，並合併後只保留科目名稱和最佳標籤兩個欄位。"""
    # 讀取兩個 CSV 檔案
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)

    # 確保必要欄位存在
    required_columns = ['系所', '科目名稱', '最佳標籤']
    for col in required_columns:
        if col not in df1.columns or col not in df2.columns:
            raise ValueError(f"檔案中缺少必要欄位：{col}")

    # 篩選最佳標籤為 '高' 的資料
    high_df1 = df1[df1['最佳標籤'] == '高']
    high_df2 = df2[df2['最佳標籤'] == '高']

    # 合併兩個篩選後的資料
    combined_df = pd.concat([high_df1, high_df2], ignore_index=True)

    # 只保留科目名稱和最佳標籤兩個欄位
    combined_df = combined_df[['系所', '科目名稱', '最佳標籤']].sort_values('系所')

    # 儲存合併後的資料
    combined_df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"已將篩選並合併後的資料儲存至 {output_file}")

# 使用範例
file1 = 'all_predictions_0113-1.csv'  # 替換為您的第一個 CSV 檔案路徑
file2 = 'all_predictions_0113-2.csv'  # 替換為您的第二個 CSV 檔案路徑
output_file = 'high_labels_combined.csv'  # 合併後的輸出檔案

filter_and_combine_high_labels(file1, file2, output_file)

已將篩選並合併後的資料儲存至 high_labels_combined.csv
